# Online ML bias-correction analysis
Compare CTRL, UNET, and UNETXTR with a common reference over the configured 90-day period. Reusable calculations live in `src/`; this notebook records the scientific workflow and products.

## 1. Configuration and paths

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from ml_implement_paper/ or its notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src import io as data_io
from src import metrics, plotting, preprocessing

config = data_io.load_config(PROJECT_ROOT / 'config' / 'paths.yaml')
paths = data_io.output_paths(config)
analysis = config['analysis']
paths

## 2. Load CTRL, UNET, UNETXTR, and reference data

In [ ]:
datasets = data_io.open_analysis_datasets(config)
datasets = {
    name: data_io.rename_available_variables(dataset, config['variables'])
    for name, dataset in datasets.items()
}
{name: dict(dataset.sizes) for name, dataset in datasets.items()}

## 3–5. Match timestamps, interpolate pressure levels, and perform QC

In [ ]:
datasets = {
    name: preprocessing.subset_time(dataset, analysis['start_date'], analysis['end_date'])
    for name, dataset in datasets.items()
}
datasets = preprocessing.match_common_times(datasets)
datasets = {name: preprocessing.daily_mean(dataset) for name, dataset in datasets.items()}

atmospheric = analysis['atmospheric_variables']
tendencies = analysis['tendency_variables']
processed = {}
for name, dataset in datasets.items():
    fields = atmospheric + [field for field in tendencies if field in dataset]
    processed[name] = preprocessing.interpolate_dataset_fields(
        dataset, fields, analysis['pressure_levels_hpa'],
        level_dim=analysis['level_dimension'],
        pressure_name=analysis.get('pressure_coordinate'),
        hybrid_names=analysis.get('hybrid_pressure'),
    )
qc = {name: preprocessing.basic_qc(dataset, atmospheric) for name, dataset in processed.items()}
qc

In [ ]:
reference = processed.pop('reference')
experiments = processed
area = preprocessing.grid_cell_weights(reference, analysis['area_variable'])
latitude = reference['lat']
land_fraction = reference.get(analysis['land_fraction_variable'])
assert list(experiments) == ['CTRL', 'UNET', 'UNETXTR']

## 6. Figure 1 — time evolution of atmospheric RMSE

In [ ]:
figure1 = metrics.global_rmse_diagnostic(experiments, reference, atmospheric, area)
data_io.save_dataset(figure1, paths['processed'] / 'global_rmse_timeseries.nc')
plotting.plot_rmse_timeseries(figure1, atmospheric, paths['figures'] / 'fig01_rmse_timeseries.png')

## 7. Figure 2 — vertical RMSE ratio

In [ ]:
figure2 = metrics.vertical_rmse_diagnostic(experiments, reference, atmospheric, area)
data_io.save_dataset(figure2, paths['processed'] / 'vertical_rmse_profiles.nc')
plotting.plot_vertical_rmse_ratio(figure2, atmospheric, paths['figures'] / 'fig02_vertical_rmse_ratio.png')

## 8. Figure 3 — spatial error reduction

In [ ]:
selections = {name: tuple(value) for name, value in analysis['spatial_selections'].items()}
figure3 = metrics.spatial_reduction_diagnostic(
    experiments, reference, selections, area, latitude, land_fraction,
    minimum_control_rmse=analysis['minimum_control_rmse'],
)
data_io.save_dataset(figure3, paths['processed'] / 'spatial_error_reduction.nc')
plotting.plot_spatial_error_reduction(figure3, list(selections), paths['figures'] / 'fig03_error_reduction.png')

## 9. Figure 4 — physical stability and ML forcing

In [ ]:
figure4 = metrics.stability_diagnostic(experiments, area, tendencies)
data_io.save_dataset(figure4, paths['processed'] / 'stability_timeseries.nc')
plotting.plot_stability(figure4, paths['figures'] / 'fig04_stability.png')

## 10–11. Summary statistics and saved products

In [ ]:
rmse_summary = figure1['rmse'].mean('time').to_dataframe(name='mean_daily_rmse').reset_index()
improvement_summary = figure1['improvement_percent'].mean('time').to_dataframe(name='mean_improvement_percent').reset_index()
regional_summary = figure3['regional_mean_reduction'].to_dataframe(name='regional_mean_rmse_reduction').reset_index()
summary_path = paths['tables'] / 'summary_metrics.csv'
pd.concat({
    'rmse': rmse_summary,
    'improvement': improvement_summary,
    'regional_reduction': regional_summary,
}, names=['diagnostic']).to_csv(summary_path, index=True)
print(rmse_summary.to_string(index=False))
print(f'Products saved under {paths["root"]}')